# Orcrist, area A (metrics): explainability

This notebook opens up Orcrist, the conditional VAE shared by all the services of an app, to see what it learned and why it raises and explains each alarm. It is the companion of `pca-B0.ipynb`, with the same protocol and the same figures where the two models can be compared, plus the parts only Orcrist has: the condition, the latent state, the decoder's deviation and the state signal.

Every number comes from `anomaly.models.evaluate.compute`, the protocol that writes `results/orcrist-<version>-B0/<app>/`. Set at the top:
- `APP`: `"hotel"` or `"social"`;
- `VERSION`: one of the versions in `orcrist.VERSIONS` (AA, AB, AC, AD);
- `FOLD`: `"dev"` while the versions are being compared, `"test"` only for the version finally chosen;
- `EXAMPLE` and `SERVICE`: the fault run of figures 4-6 and the service figures 2 and 4 follow;
- `COMPARE`: figure 11 refits every version to follow the example's target in each (about 30 s per version).

Figures:
- 1-3, the model: what the condition learned, what the latent state encodes, whether the decoder's deviation can be trusted as a normal range;
- 4-7, one fault run: observed and expected, the alarm map, the explained ranking, state against coherence;
- 8, whether the explanation points at the feature that really moved (synthetic shifts on healthy steps, against the PCA);
- 9-11, the fold: separation without a threshold, per scenario, and the versions side by side.

Terms used in the figures:
- **degree**: the service's score (sum over features of ((x − μ) / σ)², persistent over 2 steps) divided by its threshold, the largest healthy score of train + val; above 1, the service is in alarm;
- **normal range**: expected ± 2σ, the decoder's mean and deviation, in real units; σ never goes below 0.1 in log1p units;
- **state**: the KL of the step's latent posterior from the prior, over its healthy maximum; above 1, the service is in a state training never saw;
- **rules**: restart and absence; **reference system**: the model with both rules, `model+restart+absence`.

In [ ]:
import sys
import warnings
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from anomaly import build, inputs
from anomaly.models import evaluate as ev, orcrist
from anomaly.models.orcrist import model as om

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40)

APP = "hotel"  # "hotel" or "social"
VERSION = "AC"  # a key of orcrist.VERSIONS
FOLD = "dev"  # "dev" while the versions are compared; "test" only for the version finally chosen
DATA = build.DATASET / "built"
# The fault run of figures 4-6 (a run of FOLD), and the service figures 2 and 4 follow (None: the first ranked at the first alarm)
EXAMPLE = {("hotel", "dev"): "dev/mem_stress_reservation_hotel_res__r3",
           ("social", "dev"): "dev/net_corrupt_media_social_network__r2",
           ("hotel", "test"): "test/network_loss_hotel_res__r1",
           ("social", "test"): "test/port_misconfig_user_social_network__r1"}[APP, FOLD]
SERVICE = None
COMPARE = False  # figure 11: refit every version and follow the example's target in each (about 30 s per version)

res = ev.compute(f"orcrist-{VERSION}", APP, DATA)
base = ev.compute("pca", APP, DATA)  # the PCA baseline on the same data, for figures 8 and 9
m = res.model
RUNS = {r.id: r for r in res.runs}
SERVICES = res.runs[0].services
FEATURES = list(m.features)
FULL = ev.VARIANTS[-1]  # the reference system
TYPE = [om.service_type(s) for s in SERVICES]
print(f"{APP}, orcrist-{VERSION} {orcrist.VERSIONS[VERSION]}: {len(res.runs)} runs, {len(SERVICES)} services; best epoch "
      f"{m.params['best_epoch']} of {m.params['epochs']}, {m.params['parameters']} parameters, "
      f"{m.params['active_dims']} active latent dimensions of {om.K}")

In [ ]:
# Colours: categorical slots in a fixed order (as in pca-B0); blue-grey-red centred on the threshold for the degree
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, AQUA, YELLOW, RED, NEUTRAL = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948", "#f0efec"
VARIANT_COLOUR = dict(zip(ev.VARIANTS, (BLUE, ORANGE, AQUA)))
TYPE_COLOUR = dict(zip(om.TYPES, (BLUE, ORANGE, AQUA, YELLOW)))
DIVERGING = LinearSegmentedColormap.from_list("threshold", [BLUE, NEUTRAL, RED])
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#f2f7fd", "#cde2fb", "#2a78d6", "#104281"])
SEQUENTIAL_BAD = LinearSegmentedColormap.from_list("bad", ["#fdf4ef", "#f9cdb8", "#eb6834", "#8a3110"])  # darker = worse
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "axes.edgecolor": INK2,
    "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": .6, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False, "lines.linewidth": 2,
    "figure.dpi": 110})


def minutes(r):
    """Minutes from the injection (fault and noop runs) or from the first step (clean sessions)."""
    return (r.t - (r.t_inject or r.t[0])) / 60


def first_alarm(r, v=FULL):
    """Step of the first alarm that starts after the injection, or None."""
    first = ev.starts(r.alarm[v]).any(1) & r.fault
    return int(first.argmax()) if first.any() else None


def reference(r):
    """The healthy steps the figures compare with. On test: the protocol's healthy inferences. Dev has none, so on dev:
    the loaded steps of the val sessions, which stopped the training and set the thresholds but never trained the weights."""
    if FOLD == "test":
        return r.healthy if r.fold == "test" else np.zeros_like(r.on)
    return r.on if r.role == "val" else np.zeros_like(r.on)


def in_model(model):
    """[S, F]: the cells a model scores (Orcrist and PCA both leave out cells flat or mostly missing in training)."""
    if hasattr(model, "cols") and np.ndim(model.cols) == 2:
        return model.cols
    return np.array([s.cols if s is not None else np.zeros(len(FEATURES), bool) for s in model.services])


REF = "healthy inferences, test fold" if FOLD == "test" else "val sessions (healthy reference)"
r = RUNS[EXAMPLE]
k = first_alarm(r)
assert k is not None, f"{EXAMPLE} raises no alarm after the injection: pick a found run from the table of section 0"
order = ev.rank(r.norm[k], r.rule[FULL][k])
SVC = SERVICES.index(SERVICE) if SERVICE else int(order[0])
print(f"{EXAMPLE}: target {[SERVICES[i] for i in r.targets]}, first alarm {minutes(r)[k]:.1f} min after the injection, "
      f"first ranked {SERVICES[order[0]]}; figures 2 and 4 follow {SERVICES[SVC]}")

## 0. Runs you can pick

The fault runs of the fold with what the reference system did on each: found (an alarm starts after the injection), minutes to that alarm, whether an accepted target is first or in the first three, and the five services ranked at it with their degree. Any found run can be the `EXAMPLE`.

In [ ]:
f = res.faults[res.faults["fold"] == FOLD]
cols = ["run", "scenario", "targets", f"{FULL} found", f"{FULL} minutes", f"{FULL} top@1", f"{FULL} top@3", "top 5 at the first alarm"]
f[cols].rename(columns=lambda c: c.replace(f"{FULL} ", "")).set_index("run")

## 1. What the condition learned

**What is plotted.** Each service's condition vector e_s (4 numbers), coloured by its technology. Left: the vectors projected on their two main directions. Right: the cosine similarity of every pair, services grouped by technology. With the type + instance condition (AB, AC), the table below splits each vector into its technology part and its own part.

**How to read it.** The condition is how the shared network knows which service it is reading. Services close together are treated alike; a block of high similarity along the diagonal means a technology behaves as one family (the idle mongos, the memcacheds). A large instance share means the service departs from its technology, typically a busy store or an application service with its own role. Names printed on top of each other belong to services with almost the same vector: the heatmap tells them apart. AD has no learned vector (a one-hot): the section prints a note instead.

In [ ]:
if m.net.kind == "onehot":
    print(f"orcrist-{VERSION} reads a one-hot of the service: there is no learned vector to draw. Set VERSION to AA, AB or AC.")
else:
    E = np.array([res.card[s]["condition"] for s in SERVICES])
    C = E - E.mean(0)
    _, sv, Vt = np.linalg.svd(C, full_matrices=False)
    P2, share = C @ Vt[:2].T, sv**2 / (sv**2).sum()
    U = E / np.linalg.norm(E, axis=1, keepdims=True)
    idx = sorted(range(len(SERVICES)), key=lambda i: (om.TYPES.index(TYPE[i]), SERVICES[i]))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.8), gridspec_kw={"width_ratios": [1, 1.15]})
    for t, c in TYPE_COLOUR.items():
        sel = [i for i in range(len(SERVICES)) if TYPE[i] == t]
        if sel:
            a1.scatter(P2[sel, 0], P2[sel, 1], s=70, color=c, label=t, edgecolor=SURFACE, lw=2, zorder=3)
    for i, s in enumerate(SERVICES):
        a1.annotate(s, P2[i], xytext=(6, 3), textcoords="offset points", fontsize=8, color=INK2)
    a1.set(xlabel=f"direction 1 ({100 * share[0]:.0f}% of the spread)", ylabel=f"direction 2 ({100 * share[1]:.0f}%)")
    a1.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=4)
    a1.set_title("Condition vectors, projected", pad=26)
    im = a2.imshow((U @ U.T)[np.ix_(idx, idx)], cmap=DIVERGING, vmin=-1, vmax=1)
    names = [SERVICES[i] for i in idx]
    a2.set_xticks(range(len(idx)), names, rotation=90, fontsize=8)
    a2.set_yticks(range(len(idx)), names, fontsize=8)
    for lab, i in zip(a2.get_yticklabels(), idx):
        lab.set_color(TYPE_COLOUR[TYPE[i]])
    a2.grid(False)
    a2.set_title("Cosine similarity, grouped by technology")
    fig.colorbar(im, ax=a2, fraction=.046, pad=.02, label="cosine similarity")
    fig.tight_layout()
    plt.show()
    if m.net.kind == "type+instance":
        ty = m.net.tech.weight.detach().numpy()[[om.TYPES.index(t) for t in TYPE]]
        inst = m.net.instance.weight.detach().numpy()
        nt, ni = np.linalg.norm(ty, axis=1), np.linalg.norm(inst, axis=1)
        display(pd.DataFrame({"type": TYPE, "|e_type|": nt.round(2), "|e_instance|": ni.round(2),
                              "instance share %": (100 * ni / (nt + ni)).round()}, index=SERVICES)
                .sort_values(["type", "instance share %"]))

## 2. What the latent state encodes

**What is plotted.** Left: the mean KL per latent dimension and service on the training steps; a dimension the model uses carries information above the prior (above 0.01 it counts as active). Right: for one service, the posterior mean of its most used dimension at every loaded step of the val sessions, against the load generator's rate at that step. The table below gives, per service and active dimension, the rank correlation of z with the rate.

**How to read it.** The decoder already knows the service, so z only has to carry what changes from step to step. If one dimension tracks the rate closely, the latent state has learned the load, and the score asks whether the service's metrics agree with that load. Dimensions with KL near 0 are unused (posterior collapse on that dimension); all dimensions near 0 would mean the model ignores z and degenerates into a per-service mean and deviation.

In [ ]:
KL = np.array([res.card[s]["kl_per_dim"] for s in SERVICES])  # [S, K], mean over the training steps
dims = np.argsort(-KL.mean(0))
active = [d for d in dims if KL[:, d].mean() > .01]
val = [rr for rr in res.runs if rr.role == "val"]
rate = lambda rr: np.load(DATA / APP / f"{rr.id}.npz")["rate"]
Z = np.concatenate([m.latent(rr.P)[0][rr.on] for rr in val])  # [N, S, K]
R = np.concatenate([rate(rr)[rr.on] for rr in val])
d0 = dims[0]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, .26 * len(SERVICES) + 2.5), gridspec_kw={"width_ratios": [1, 1.4]})
im = a1.imshow(KL, cmap=SEQUENTIAL, aspect="auto", vmin=0)
a1.set_yticks(range(len(SERVICES)), SERVICES, fontsize=8)
a1.set_xticks(range(om.K), [f"z{d + 1}" for d in range(om.K)])
a1.grid(False)
a1.set_title("Mean KL per dimension, training")
fig.colorbar(im, ax=a1, fraction=.05, pad=.03, label="KL (nats)")
a2.scatter(R, Z[:, SVC, d0], s=8, color=BLUE, alpha=.4, lw=0)
a2.set(xlabel="load generator rate (req/s)", ylabel=f"posterior mean of z{d0 + 1}",
       title=f"{SERVICES[SVC]}: z{d0 + 1} against the load, val sessions ({len(R)} steps)")
fig.tight_layout()
plt.show()
rank = pd.Series(R).rank()  # Spearman's rho: the Pearson correlation of the ranks
rho = {f"z{d + 1}": [pd.Series(Z[:, i, d]).rank().corr(rank) for i in range(len(SERVICES))] for d in active}
pd.DataFrame(rho, index=SERVICES).round(2).assign(type=TYPE)

## 3. Can the normal range be trusted?

**What is plotted.** On the healthy reference steps (see `REF` in the title), the observed value of every scored cell against the decoder's expected value and deviation. Left: the residual in deviations, (x − μ) / σ, pooled over all cells, with the standard normal curve for comparison. Right: per service and feature, the share of steps whose observed value falls inside the normal range μ ± 2σ, centred on 95% (grey): red cells are under 95% (the range is too narrow there and would cry wolf), blue cells over it (the range is wider than the data needs, usually because σ sits at its floor). Blank cells are out of the model. The table below gives the share of steps where σ sits at its floor, per feature.

**How to read it.** The MAS receives the normal range as the explanation of an alarm, so it should hold about 95% of healthy values. A residual histogram narrower than the normal curve means the floor dominates: steady cells whose real noise is below 0.1 in log1p units. A red cell is a service and feature where healthy values often leave the range: a source of false alarms and of misleading explanations.

In [ ]:
cells = in_model(m)
inside, count, resid, floor = (np.zeros(cells.shape), np.zeros(cells.shape), [], np.zeros(cells.shape))
for rr in res.runs:
    ref = reference(rr)
    if not ref.any():
        continue
    x, e, sd = rr.P[ref], rr.expected[ref], rr.sd[ref]
    ok = np.isfinite(x) & np.isfinite(e) & cells[None]
    inside += (ok & (np.abs(x - e) <= 2 * sd)).sum(0)
    count += ok.sum(0)
    floor += (ok & (sd <= 1.02 * om.SIGMA_MIN)).sum(0)
    resid.append(((x - e) / sd)[ok])
cover = np.where(count > 0, 100 * inside / np.maximum(count, 1), np.nan)
z = np.concatenate(resid)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, .28 * len(SERVICES) + 2.6), gridspec_kw={"width_ratios": [1, 1.5]})
a1.hist(np.clip(z, -6, 6), bins=np.linspace(-6, 6, 61), density=True, color=BLUE, alpha=.8, edgecolor=SURFACE, lw=.5,
        label=f"healthy residuals ({len(z)} cells)")
g = np.linspace(-6, 6, 300)
a1.plot(g, np.exp(-g**2 / 2) / np.sqrt(2 * np.pi), color=INK, lw=1.4, label="standard normal")
for s in (-2, 2):
    a1.axvline(s, color=INK2, ls="--", lw=1)
a1.set(xlabel="(x − μ) / σ, clipped at ±6", ylabel="density",
       title=f"{100 * (np.abs(z) <= 2).mean():.1f}% inside ±2σ ({REF})")
a1.legend(loc="upper left")
im = a2.imshow(cover, cmap=LinearSegmentedColormap.from_list("cov", [RED, NEUTRAL, BLUE]),
               norm=TwoSlopeNorm(95, 50, 100), aspect="auto")
for (a, b), v in np.ndenumerate(cover):
    if np.isfinite(v):
        a2.text(b, a, f"{v:.0f}", ha="center", va="center", fontsize=7, color=INK if 70 < v < 99.5 else SURFACE)
a2.set_yticks(range(len(SERVICES)), SERVICES, fontsize=8)
a2.set_xticks(range(len(FEATURES)), FEATURES, rotation=45, ha="right", fontsize=8)
a2.grid(False)
a2.set_title("Healthy values inside μ ± 2σ (%)")
fig.colorbar(im, ax=a2, fraction=.04, pad=.02, ticks=[50, 80, 95, 100])
fig.tight_layout()
plt.show()
pd.DataFrame({"steps with σ at its floor %": (100 * floor.sum(0) / np.maximum(count.sum(0), 1)).round(1),
              "cells inside μ ± 2σ %": (100 * inside.sum(0) / np.maximum(count.sum(0), 1)).round(1)}, index=FEATURES)

## 4. Observed, expected and the normal range, around the injection

**What is plotted.** Top: the feature with the largest share of the service's score after the injection, observed (black) and expected by the decoder (dashed blue), with the normal range μ ± 2σ (blue band), in its real unit (memory is a change per 30 s step; symmetric log scale). Bottom: the service's degree (orange) and state (green) on a log scale; above the dashed line the service is in alarm (degree) or in a state never seen in training (state). The red line is the injection; the grey band is the warm-up, left out of fitting and evaluation.

**How to read it.** Before the fault the observed curve stays inside the band. When it leaves, the distance in bands is the residual the score measures, and its feature is the explanation. If the degree rises while the state stays below 1, the latent state is familiar (the service sees a load it knows) but its metrics no longer agree with it: a broken relation, typical of a fault. A state above 1 says the service itself is somewhere new, which can also be an unusual load.

In [ ]:
i = SVC
j = int(np.argmax(r.r2[r.fault][:, i].sum(0)))
f = FEATURES[j]
tm = minutes(r)
lo, hi = (inputs.to_units(r.expected[:, i, j] + n * r.sd[:, i, j], f) for n in (-2, 2))
lo = lo if f.startswith("mem") else np.maximum(lo, 0)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
a1.fill_between(tm, lo, hi, color=BLUE, alpha=.15, lw=0, label="normal range, μ ± 2σ")
a1.plot(tm, inputs.to_units(r.expected[:, i, j], f), color=BLUE, lw=1.4, ls="--", label="expected (μ)")
a1.plot(tm, inputs.to_units(r.P[:, i, j], f), color=INK, lw=1.6, label="observed")
a1.set_yscale("symlog", linthresh=1)
a1.set(ylabel=f"{f} ({inputs.UNITS[f][1]}, symmetric log)", title=f"{SERVICES[i]}, {f}: observed, expected and normal range")
a1.legend(loc="upper left", ncol=3)
a2.plot(tm, np.maximum(r.norm[:, i], 1e-2), color=ORANGE, lw=1.6, label="degree (score / threshold)")
a2.plot(tm, np.maximum(r.state[:, i], 1e-2), color=AQUA, lw=1.6, label="state (KL / healthy max)")
a2.axhline(1, color=INK2, ls="--", lw=1)
a2.set_yscale("log")
a2.set(ylabel="ratio to the healthy maximum", xlabel="minutes from the injection")
a2.legend(loc="upper left", ncol=2)
for a in (a1, a2):
    a.axvspan(tm[0] - .5, 0, color=GRID, alpha=.7, lw=0)
    a.axvline(0, color=RED, lw=1.2)
a1.text(0, a1.get_ylim()[1], " injection", va="top", color=RED, fontsize=9)
fig.tight_layout()
plt.show()

## 5. Alarm map of the run

**What is plotted.** Every service (rows) at every 30 s step (columns): colour is the degree on a log scale, grey at the threshold, blue below it (healthy), red above it (alarm). Crosses mark restarts, squares mark absence (the rules). The black line is the injection, the dotted line the first alarm; the hatched band is the warm-up. The target's name ends with ◀.

**How to read it.** The first red cell after the injection is the first alarm, and the ranking of figure 6 is taken at that column. A row that is red across the whole run, before the injection too, is a service whose healthy behaviour the model has not captured: a false-alarm source to look at in figure 3.

In [ ]:
edges = np.r_[tm[0] - .5, tm]
fig, ax = plt.subplots(figsize=(11, .3 * len(SERVICES) + 1.8))
mesh = ax.pcolormesh(edges, np.arange(len(SERVICES) + 1), np.log10(np.clip(r.norm, 1e-2, 1e2)).T, cmap=DIVERGING,
                     norm=TwoSlopeNorm(0, -2, 2), shading="flat")
for rule, marker in ((r.restart, "x"), (r.absent, "s")):
    tt, ss = np.where(rule & (r.on | r.fault)[:, None])
    style = {"color": INK} if marker == "x" else {"facecolors": "none", "edgecolors": INK}
    ax.scatter(tm[tt] - .25, ss + .5, marker=marker, s=22, lw=1, **style)
ax.axvspan(edges[0], 0, facecolor="none", edgecolor=INK2, hatch="///", lw=0)
ax.axvline(0, color=INK, lw=1.5)
ax.axvline(tm[k], color=INK, lw=1, ls=":")
ax.set_yticks(np.arange(len(SERVICES)) + .5, [s + (" ◀" if n in r.targets else "") for n, s in enumerate(SERVICES)])
ax.invert_yaxis()
ax.grid(False)
ax.set_xlabel("minutes from the injection")
ax.set_title(f"{EXAMPLE}: degree per service", pad=26)
cb = fig.colorbar(mesh, ax=ax, ticks=[-2, -1, 0, 1, 2], pad=.01)
cb.ax.set_yticklabels(["0.01", "0.1", "1 (threshold)", "10", "100+"])
ax.legend(handles=[Line2D([], [], marker="x", ls="", color=INK, label="restart"),
                   Line2D([], [], marker="s", ls="", mfc="none", mec=INK, label="absent"),
                   Line2D([], [], color=INK, lw=1.5, label="injection"),
                   Line2D([], [], color=INK, lw=1, ls=":", label="first alarm")],
          loc="lower left", bbox_to_anchor=(0, 1.0), ncol=4)
fig.tight_layout()
plt.show()

## 6. Why each service ranks where it does

**What is plotted.** The first five services of the ranking at the first alarm, in order. Each bar splits the service's score into the shares of its feature groups (CPU, memory, threads, network); on the right, the degree, the rule met if any, and the largest feature. The table below is the explanation the MAS receives for the services in alarm, the same record as `localization.jsonl`: per feature its share, direction, observed and expected value and normal range, in real units.

**How to read it.** The bar says which kind of metric made the service anomalous; the table says by how much, against what this service's own load would explain. A service meeting a rule ranks first whatever its score; its bar can be empty, because a pod that is gone has no metrics to explain.

In [ ]:
GROUPS = {"CPU": ("cpu_user", "cpu_system"), "memory": ("mem_working_set", "mem_rss", "mem_cache"),
          "threads": ("threads",), "network": ("rx_bytes", "tx_bytes", "rx_packets", "tx_packets")}
GROUP_COLOUR = dict(zip(GROUPS, (BLUE, ORANGE, AQUA, YELLOW)))
top = order[:5]
fig, ax = plt.subplots(figsize=(10, 3.4))
for row, s in enumerate(top):
    c = r.r2[k, s]
    left = 0.0
    for g, feats in GROUPS.items():
        share = 100 * sum(c[FEATURES.index(f)] for f in feats) / c.sum() if c.sum() else 0.0
        ax.barh(row, share, left=left, color=GROUP_COLOUR[g], height=.6, edgecolor=SURFACE, lw=1)
        left += share
    best = FEATURES[int(np.argmax(c))] if c.sum() else "none"
    ax.text(104, row, f"degree {r.norm[k, s]:.1f}{ev.tags(r, k, s)}; largest: {best}", va="center", fontsize=9, color=INK2)
ax.set_yticks(range(len(top)), [SERVICES[s] + (" ◀ target" if s in r.targets else "") for s in top])
ax.invert_yaxis()
ax.set_xlim(0, 200)
ax.set_xticks([0, 25, 50, 75, 100])
ax.spines["bottom"].set_bounds(0, 100)
ax.grid(axis="y", visible=False)
ax.set_xlabel("share of the service's score (%)")
ax.set_title(f"Ranking at the first alarm, {minutes(r)[k]:.1f} min after the injection", pad=26)
ax.legend(handles=[Patch(color=c, label=g) for g, c in GROUP_COLOUR.items()], loc="lower left", bbox_to_anchor=(0, 1.0), ncol=4)
fig.tight_layout()
plt.show()
rows = [{"service": SERVICES[s], "degree": round(float(r.norm[k, s]), 2), "state": round(float(r.state[k, s]), 2), **w}
        for s in top if r.alarm[FULL][k, s] for w in ev.why(r, k, s, FEATURES)]
pd.DataFrame(rows).set_index(["service", "degree", "state"])

## 7. State against coherence

**What is plotted.** Every step as a point: horizontal, its degree (how badly the features fit the decoder); vertical, its state (how unusual the latent state is). Blue: all services on the healthy reference steps. Orange: the accepted targets at every step after the injection, in the fault runs of the fold. The dashed lines are 1 on each axis; the table below gives the share of each group in each quadrant.

**How to read it.** Bottom-right: the features break their usual relation while the service's state is familiar, the signature of a fault. Top-right: both, a fault that also pushes the service somewhere new, or a load never seen. Top-left: a new state whose features still agree with each other, which is what an unusual load looks like. If the healthy points sit mostly bottom-left and the faults spread right, the state adds little to the alarm; if some faults only go up, the state could find them.

In [ ]:
def pts(sel):
    return np.clip(np.concatenate([a.ravel() for a in sel]) if sel else np.array([]), 1e-3, 1e3)


hx, hy, fx, fy = [], [], [], []
for rr in res.runs:
    ref = reference(rr)
    if ref.any():
        hx.append(rr.norm[ref]), hy.append(rr.state[ref])
    if rr.fold == FOLD and rr.fault.any() and rr.targets:
        fx.append(rr.norm[rr.fault][:, rr.targets]), fy.append(rr.state[rr.fault][:, rr.targets])
hx, hy, fx, fy = pts(hx), pts(hy), pts(fx), pts(fy)
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.scatter(hx, hy, s=5, color=BLUE, alpha=.2, lw=0, label=f"{REF}, all services ({len(hx)})")
ax.scatter(fx, fy, s=12, color=ORANGE, alpha=.7, lw=0, label=f"targets after the injection, {FOLD} faults ({len(fx)})")
ax.axvline(1, color=INK2, ls="--", lw=1)
ax.axhline(1, color=INK2, ls="--", lw=1)
ax.set(xscale="log", yscale="log", xlabel="degree (score / threshold)", ylabel="state (KL / healthy max)")
ax.legend(loc="upper left")
ax.set_title("State against coherence")
fig.tight_layout()
plt.show()
quad = lambda x, y: {"both below 1": (x <= 1) & (y <= 1), "degree only": (x > 1) & (y <= 1),
                     "state only": (x <= 1) & (y > 1), "both above 1": (x > 1) & (y > 1)}
pd.DataFrame({REF: {q: 100 * v.mean() for q, v in quad(hx, hy).items()},
              f"targets after the injection ({FOLD})": {q: 100 * v.mean() for q, v in quad(fx, fy).items()}}).round(1)

## 8. Does the explanation point at the feature that moved?

**What is plotted.** A controlled experiment on healthy data. From the loaded steps of the val sessions, 400 steps are drawn; for each feature in turn, every service's value of that feature is shifted by k times its healthy deviation (its standard deviation over the training steps, in log1p units), all else untouched, and both models score the result. Left column: the share of shifted cells whose largest score share is the shifted feature (the explanation names the right feature). Right column: the median share of the score's increase that lands on the other features (the residual smeared away from the cause). Top row: Orcrist; bottom row: the PCA baseline on the same steps. Darker is better on the left (blue) and worse on the right (orange). Negative k lowers the feature (clipped at 0 for features that cannot be negative).

**How to read it.** Each model scores one service at one step on its own, so shifting a feature in every service at once is the same as shifting it in each service separately. A good explanation has a high left column and a low right one. Orcrist's encoder sees the shifted value, so z can move to partly follow it: that shows up as smearing, and is the limit a masked-feature explanation would remove. A feature with a low hit rate at large k is one whose alarms the MAS should not take at face value.

In [ ]:
KS = (-4, -2, 2, 4, 8)
pool = np.concatenate([rr.P[rr.on] for rr in res.runs if rr.role == "val"])
X0 = pool[np.random.default_rng(0).choice(len(pool), min(400, len(pool)), replace=False)]
cell_sd = np.nanstd(np.concatenate([rr.P[rr.on] for rr in res.runs if rr.role == "train"]), 0)  # [S, F]


def faithfulness(model):
    """(hit %, smear %), both [F, len(KS)]."""
    r0, cols = model.score(X0)[0], in_model(model)
    hit, smear = np.full((len(FEATURES), len(KS)), np.nan), np.full((len(FEATURES), len(KS)), np.nan)
    for j, f in enumerate(FEATURES):
        for c, kk in enumerate(KS):
            X1 = X0.copy()
            X1[:, :, j] += kk * cell_sd[:, j]
            if not f.startswith("mem"):
                X1[:, :, j] = np.maximum(X1[:, :, j], 0)
            r1 = model.score(X1)[0]
            gain = r1.sum(2) - r0.sum(2)
            other = gain - (r1[:, :, j] - r0[:, :, j])
            ok = np.isfinite(X0[:, :, j]) & cols[None, :, j] & (cell_sd[None, :, j] > 1e-6) & (gain > 1e-3)
            if ok.any():
                hit[j, c] = 100 * (r1.argmax(2) == j)[ok].mean()
                smear[j, c] = 100 * np.median((other / gain)[ok])
    return hit, smear


fig, axes = plt.subplots(2, 2, figsize=(12, 9.5), sharex=True, sharey=True)
for row, (name, model) in enumerate(((f"Orcrist {VERSION}", m), ("PCA", base.model))):
    for col, (what, M) in enumerate(zip(("names the shifted feature (%)", "smeared on other features (%)"), faithfulness(model))):
        a = axes[row, col]
        a.imshow(M, cmap=(SEQUENTIAL, SEQUENTIAL_BAD)[col], vmin=0, vmax=100, aspect="auto")
        for (y, x), v in np.ndenumerate(M):
            if np.isfinite(v):
                a.text(x, y, f"{v:.0f}", ha="center", va="center", fontsize=8, color=SURFACE if v > 55 else INK)
        a.set_xticks(range(len(KS)), [f"{kk:+d} sd" for kk in KS])
        a.set_yticks(range(len(FEATURES)), FEATURES, fontsize=8)
        a.grid(False)
        a.set_title(f"{name}: {what}")
fig.tight_layout()
plt.show()

## 9. Healthy and fault inferences, without a threshold

**What is plotted.** Left: for every inference, the highest degree over the services, as a cumulative distribution (log scale): blue the healthy reference, orange the fault inferences of the fold. The dashed line is the threshold. Right: what happens if every threshold is multiplied by a factor c, share of healthy inferences in alarm against share of fault inferences in alarm, for Orcrist and for the PCA; the dot is c = 1. Rules are left out: this is the model alone.

**How to read it.** The further apart the two curves, the better the model separates healthy from faulty steps whatever the threshold, and the area under the right curve (AUC) is that separation in one number. On dev the healthy reference is the val sessions, which are part of the threshold's maximum, so their degree never exceeds 1 and the curves are optimistic at c ≥ 1; the comparison between the two models stays fair, since both are built the same way.

In [ ]:
def separation(rs):
    h = np.concatenate([rr.norm[reference(rr)].max(1) for rr in rs.runs if reference(rr).any()])
    fl = np.concatenate([rr.norm[rr.fault].max(1) for rr in rs.runs if rr.fold == FOLD and rr.fault.any()])
    return h, fl


def roc(h, fl):
    hs, fs = np.sort(h), np.sort(fl)
    cs = np.r_[0, np.unique(np.r_[hs, fs])]
    fpr = 1 - np.searchsorted(hs, cs, side="right") / len(hs)
    tpr = 1 - np.searchsorted(fs, cs, side="right") / len(fs)
    return fpr, tpr, -np.trapezoid(tpr, fpr)


h, fl = separation(res)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.4))
for v, c, lab in ((h, BLUE, f"{REF} ({len(h)})"), (fl, ORANGE, f"fault inferences, {FOLD} ({len(fl)})")):
    xs = np.sort(np.clip(v, 1e-2, None))
    a1.plot(xs, np.arange(1, len(xs) + 1) / len(xs), color=c, label=lab, drawstyle="steps-post")
a1.axvline(1, color=INK2, ls="--", lw=1)
a1.set_xscale("log")
a1.set(xlabel="highest degree over the services", ylabel="share of inferences at or below",
       title=f"Orcrist {VERSION}, {FOLD} fold")
a1.legend(loc="lower right")
for name, rs, c in ((f"Orcrist {VERSION}", res, BLUE), ("PCA", base, AQUA)):
    hh, ff = separation(rs)
    fpr, tpr, auc = roc(hh, ff)
    a2.plot(fpr, tpr, color=c, label=f"{name}, AUC {auc:.3f}")
    a2.plot((hh > 1).mean(), (ff > 1).mean(), "o", color=c, ms=7)
a2.plot([0, 1], [0, 1], color=GRID, ls="--", lw=1)
a2.set(xlabel="healthy inferences in alarm", ylabel="fault inferences in alarm", xlim=(0, 1), ylim=(0, 1.02),
       title="Moving the threshold (dot: c = 1)")
a2.legend(loc="lower right")
fig.tight_layout()
plt.show()

## 10. Per scenario

**What is plotted.** Left: recall per inference, the share of fault inferences in alarm, for the model and its two rule variants. Right: localization of the reference system, the share of faults with an accepted target among the first 1, 3 and 5 services at the first alarm.

**How to read it.** Left says whether the system notices the fault and keeps noticing it; right says whether it points at the right place. A scenario with high recall and low top@1 is found but blamed on a neighbour (figure 5 shows which).

In [ ]:
sc = res.scenarios.loc[FOLD]
names = list(sc.index)
y = np.arange(len(names))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, .5 * len(names) + 2), sharey=True)
for off, v in zip((-.27, 0, .27), ev.VARIANTS):
    a1.barh(y + off, sc[f"{v} recall"], height=.26, color=VARIANT_COLOUR[v], label=v, edgecolor=SURFACE, lw=1)
for off, n, c in zip((-.27, 0, .27), ev.TOP, (BLUE, ORANGE, AQUA)):
    a2.barh(y + off, sc[f"{FULL} top@{n}"], height=.26, color=c, label=f"top@{n}", edgecolor=SURFACE, lw=1)
a1.set_yticks(y, names)
a1.invert_yaxis()
a1.set(xlabel="fault inferences in alarm (%)", xlim=(0, 100))
a2.set(xlabel="faults with an accepted target in the first k (%)", xlim=(0, 100))
a1.set_title("Recall per inference", pad=40)
a2.set_title("Localization at the first alarm, reference system", pad=40)
for a, ncol in ((a1, 2), (a2, 3)):
    a.grid(axis="y", visible=False)
    a.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=ncol)
fig.tight_layout()
plt.show()

## 11. The versions side by side

**What is plotted.** The table: the reference system of every version and of the PCA on the fold, read from `results/` (run `python -m anomaly.models.evaluate orcrist-<version> <app>` for a version that is missing). With `COMPARE = True`, the figure refits every version and draws the degree of the example's first target over the run in each.

**How to read it.** The versions differ in one choice each (see `orcrist.VERSIONS`), so a difference between two rows is the effect of that choice. These are one seed each: on dev a single fault moves a percentage by about 4 points, so small gaps are not evidence yet.

In [ ]:
keep = ["recall", "faults found %", "minutes to alarm, median", "top@1 %", "top@3 %", "top@5 %"]
if FOLD == "test":
    keep = ["F1", "precision"] + keep + [f"healthy in alarm % {s}" for s in ev.SOURCES]
table = {}
for name in ["pca"] + [f"orcrist-{v}" for v in orcrist.VERSIONS]:
    p = REPO / "results" / f"{name}-B0" / APP / "summary.csv"
    if p.exists():
        table[name] = pd.read_csv(p).set_index(["fold", "variant"]).loc[(FOLD, FULL), keep]
if COMPARE:
    tgt = r.targets[0]
    fig, ax = plt.subplots(figsize=(10, 4))
    for v, c in zip(orcrist.VERSIONS, (BLUE, ORANGE, AQUA, YELLOW)):
        rv = res if v == VERSION else ev.compute(f"orcrist-{v}", APP, DATA)
        ax.plot(tm, np.maximum({x.id: x for x in rv.runs}[EXAMPLE].norm[:, tgt], 1e-2), color=c, lw=1.5, label=f"orcrist-{v}")
    ax.plot(tm, np.maximum({x.id: x for x in base.runs}[EXAMPLE].norm[:, tgt], 1e-2), color=INK2, lw=1.2, ls="--", label="PCA")
    ax.axhline(1, color=INK2, ls=":", lw=1)
    ax.axvline(0, color=RED, lw=1.2)
    ax.set_yscale("log")
    ax.set(xlabel="minutes from the injection", ylabel="degree")
    ax.set_title(f"{EXAMPLE}: degree of {SERVICES[tgt]} in each version", pad=26)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=5)
    fig.tight_layout()
    plt.show()
pd.DataFrame(table).T

## The numbers behind the figures

The rows of `summary.csv` for this version, app and fold, one column per variant.

In [ ]:
res.summary.loc[FOLD].T